# RAG 데이터 확인용 노트북

`.chroma/`에 저장된 벡터스토어 내용을 직접 들여다보고, 검색/답변 결과를 셀 단위로 확인하기 위한 노트북입니다.
커널은 프로젝트의 `.venv`를 선택하세요.

In [13]:
import pandas as pd
from rag_core import vectorstore, embeddings, llm
from dart_parser import parse_dart_xml

pd.set_option("display.max_colwidth", 120)

## 1. 저장된 청크 전체 개요
문서(source)별로 몇 개의 청크가 들어있는지 확인합니다.

In [14]:
raw = vectorstore.get(include=["documents", "metadatas"])
df = pd.DataFrame({
    "id": raw["ids"],
    "text": raw["documents"],
    **{k: [m.get(k) for m in raw["metadatas"]] for k in ["source", "chunk_index", "section", "company", "doc_type"]},
})
print("총 청크 수:", len(df))
df.groupby("source").size().sort_values(ascending=False)

총 청크 수: 7763


source
20210317001085.xml    187
20201116001322.xml    185
20260318001585.xml    172
20200814002597.xml    168
20240313001807.xml    166
                     ... 
20220404000994.xml     27
20250324000737.xml     27
20240329001212.xml     27
20210331001087.xml     26
20260330000843.xml     26
Length: 82, dtype: int64

## 2. 특정 문서의 청크 살펴보기
`source_filter`를 원하는 파일명으로 바꿔서 실행하세요.

In [15]:
source_filter = df["source"].iloc[0] if len(df) else None
df[df["source"] == source_filter][["chunk_index", "section", "text"]]

,chunk_index,section,text
0,0,표지,(제 7기 반기)\n\n사업연도 | 2020년 01월 01일 | 부터\n2020년 06월 30일 | 까지\n\n금융위원회 | \n한국거래소 귀중 | 2020 년 8 월 14 일\n\n제출대상법인 유형 : | ...
1,1,I. 회사의 개요 > 1. 회사의 개요,"가. 회사의 법적, 상업적 명칭\n 당사의 명칭은 주식회사 삼양패키징으로 표기합니다. \n영문으로는 Samyang Packaging Corporation이라고 표기합니다.\n나. 설립일자 및 존속기간\n 당사..."
2,2,I. 회사의 개요 > 1. 회사의 개요,주소: 서울 종로구 종로33길 31 \n전화번호: (02) 740-7114\n홈페이\n지: \nhttp://samyangpackaging.co.kr/\n라. 중소기업 해당여부 \n당사는 사업보고서 제출일 현재...
3,3,I. 회사의 개요 > 1. 회사의 개요,당사는 보고서 작성 기준일 현재 당사를 포함하여 19개의 계열회사가 있습니다.\n\n구분 | 회사명 | 비 고\n1 | (주)삼양홀딩스 | 상 장\n2 | (주)삼양사\n3 | (주)삼양패키징\n4 | (주)...
4,4,I. 회사의 개요 > 1. 회사의 개요,* 신용등급체계 및 부여 의미\n\n신용등급\n체계 | 등급 정의\nAAA | 원리금 지급능력이 최상급임\nAA | 원리금 지급능력이 매우 우수하지만 AAA의 채권보다는 다소 열위임\nA | 원리금 지급능력은...
...,...,...,...
85,85,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,2. \n제\n재\n현\n황\n 등 그 밖의 사항\n가. 작성기준일 이후 발생한 주요 사항\n해\n당사항 없음\n나. 중소기업기준 검토표\n\n당사는 중소기업에 해당하지 않습니다.\n\n다. \n직접금융 자금...
86,86,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,"온실가스 배출량(tCO2e) | 에너지 사용량(TJ)\n144,902 | 2,957\n\n온실가스 저감활동으로 당사는 사업장의 저효율 에너지설비에 대한 고효율설비로 교체 작업을 진행하여 온실가스 발생에 대한 ..."
87,87,XI. 그 밖에 투자자 보호를 위하여 필요한 사항,"또한 Aseptic 음료제조시설 투자를 통하여 포장용기의 경량화를 통한 탄소저감에 기여하고 있으며, 청정연료인 LNG 보일러 전환, LED 조명 전환, 고효율 Utility 설비 교체 등의 에너지 효율화 작업..."
88,88,【 전문가의 확인 】 > 1. 전문가의 확인,해당사항 없음


## 3. 유사도 검색 테스트
질문을 바꿔가며 어떤 청크가 검색되는지, 유사도 점수는 어떤지 확인합니다.

In [18]:
question = "삼양엔씨켐 대표이사는 누구야?"
results = vectorstore.similarity_search_with_score(question, k=5)

pd.DataFrame([
    {
        "score": score,
        "source": doc.metadata.get("source"),
        "section": doc.metadata.get("section"),
        "text": doc.page_content[:200],
    }
    for doc, score in results
])

,score,source,section,text
0,0.879206,20240814001090.xml,VIII. 임원 및 직원 등에 관한 사항 > 1. 임원 및 직원 등의 현황,삼양에프앤비 대표('22~현재) | - | - | - | 2022.12.01 ~현재 | -\n정우경 | 여 | 1966년 03월 | 임원 | 미등기 | 상근 | 식품연구소장 | 서울대 식품영약학과 졸업 서울대...
1,0.988045,20210317001085.xml,VIII. 임원 및 직원 등에 관한 사항 > 1. 임원 및 직원 등의 현황,세왕금속공업㈜ 사장\n세무법인 세연 대표세무사(현)\n삼양사 사외이사('20~현재) | - | - | 계열회사 임원 | 2020.03.26\n ~현재 | 2023.03.26\n최낙현 | 남 | 1964년 03...
2,0.989390,20250813000954.xml,VIII. 임원 및 직원 등에 관한 사항 > 1. 임원 및 직원 등의 현황,- 국민리스 대리('95~'03)\n- 동우화인켐 재무팀장('03~'12)\n- 코스틸 전략재무본부 본부장('13~'19)\n- 삼양엔씨켐 상무('19~현재) | - | - | 타인 | 5년 11개월 | -\...
3,0.996387,20210517000924.xml,VIII. 임원 및 직원 등에 관한 사항 > 1. 임원 및 직원 등의 현황,"삼양화성 이사,\n삼양이노켐 이사\n 삼남석유화학 이사\n 삼양화인테크놀로지 이사 | 연세대 화학공학과 졸업 \n일리노이공과대 화학공학 석사 \n삼양사 화학그룹장( '21.1~'21.3)\n삼양사 대표이사 겸..."
4,1.005594,20210817001681.xml,VIII. 임원 및 직원 등에 관한 사항 > 1. 임원 및 직원 등의 현황,세왕금속공업(주) 사장\n세무법인 세연 대표세무사(현)\n삼양사 사외이사('20~현재) | - | - | 계열회사 임원 | 2020.03.26 ~현재 | 2023.03.26\n최낙현 | 남 | 1964년 03...


## 4. 전체 RAG 파이프라인(query.py의 ask) 실행
검색 + 프롬프트 + GPT 답변까지 한 번에 확인합니다.

In [19]:
from query import ask

print(ask(question))

삼양엔씨켐의 대표이사는 정회식입니다.


## 5. (선택) DART XML 파서 결과만 따로 확인
`ingest.py`를 거치기 전, `dart_parser`가 특정 XML 파일을 어떤 섹션들로 쪼개는지 미리 확인할 때 사용합니다.

In [10]:
from pathlib import Path

xml_path = Path("data/dart_xml/파일명.xml")  # 확인하고 싶은 파일로 변경
sections = parse_dart_xml(xml_path)

pd.DataFrame([
    {"section": d.metadata["section"], "length": len(d.page_content), "preview": d.page_content[:150]}
    for d in sections
])

FileNotFoundError: [Errno 2] No such file or directory: 'data\\dart_xml\\파일명.xml'